In [ ]:
# Colab setup — run this cell first, then run all
import subprocess, sys, os

REPO = '/content/Katabatic'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/lukebrumby/katabatic-personal.git', REPO], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Setup complete, CWD:', os.getcwd())

In [ ]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models_luke.adasyn.models import ADASYNModel

In [ ]:
ADASYN = lambda: ADASYNModel(
    sampling_strategy="auto",
    n_neighbors=5,
    random_state=42,
)

In [ ]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "car.csv"
output_path = ROOT / "discretized_data" / "car.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))

In [ ]:
# Run pipeline
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "car")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "car" / "adasyn")

pipeline = TrainTestSplitPipeline(model=ADASYN)
result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)
print(result)